In [1]:
import time
import re
import pytz
import pandas as pd
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import selenium.common.exceptions as se

#pd.set_option("display.max_colwidth", None)

TR_MONTHS = {
    "Ocak": 1, "Şubat": 2, "Mart": 3, "Nisan": 4, "Mayıs": 5, "Haziran": 6,
    "Temmuz": 7, "Ağustos": 8, "Eylül": 9, "Ekim": 10, "Kasım": 11, "Aralık": 12
}

def make_driver(detach=True, headless=False):
    chrome_options = webdriver.ChromeOptions()
    if detach:
        chrome_options.add_experimental_option("detach", True)
    if headless:
        chrome_options.add_argument("--headless=new")

    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1400,900")
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=chrome_options)
    driver.set_page_load_timeout(60)
    return driver

In [2]:
def click_if_present(driver, css, wait_sec=3):
    try:
        WebDriverWait(driver, wait_sec).until(EC.presence_of_element_located((By.CSS_SELECTOR, css)))
        els = driver.find_elements(By.CSS_SELECTOR, css)
        if els:
            el = els[0]
            if el.is_displayed():
                driver.execute_script("arguments[0].click();", el)
                return True
    except Exception:
        pass
    return False

def click_first_visible(driver, selectors, wait_sec=4):
    """
    selectors: list[str]
    Listedeki selector'lar içinde ilk görünen/clickable olanı tıklar.
    """
    for sel in selectors:
        try:
            el = WebDriverWait(driver, wait_sec).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, sel))
            )
            if el and el.is_displayed():
                driver.execute_script("arguments[0].click();", el)
                time.sleep(0.2)
                return True
        except Exception:
            pass
    return False

In [3]:
def parse_tr_date(day_str, month_str, year_str):
    day = int(re.sub(r"\D", "", day_str))
    month = TR_MONTHS.get(month_str.strip(), None)
    year = int(re.sub(r"\D", "", year_str))
    if not month:
        return None
    return datetime(year, month, day).date().isoformat()

def safe_text(el):
    try:
        return el.text.strip()
    except Exception:
        return ""

def try_find_text(parent, selectors):
    for sel in selectors:
        try:
            found = parent.find_element(By.CSS_SELECTOR, sel)
            txt = safe_text(found)
            if txt:
                return txt
        except Exception:
            pass
    return None

In [4]:
def get_product_title(driver, wait_sec=6):
    try:
        el = WebDriverWait(driver, wait_sec).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'h1[data-testid="product-title"]'))
        )
        return el.text.strip()
    except Exception:
        return None

In [5]:
def expand_product_description_if_needed(driver):
    # 1) "ÜRÜNÜN TÜM ÖZELLİKLERİ"
    click_first_visible(driver, [
        'button[data-testid="show-more-button"]',
        'button.show-more-button-show-more-button',
    ], wait_sec=3)

    # 2) "Daha Fazla Göster" (description içinde)
    click_first_visible(driver, [
        'div.product-info-content button.show-more-button',
        'button.show-more-button',
    ], wait_sec=2)

In [6]:
def get_product_description(driver, wait_sec=6):
    try:
        WebDriverWait(driver, wait_sec).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.product-info-content"))
        )

        # Expand dene
        expand_product_description_if_needed(driver)

        # Paragrafları topla
        ps = driver.find_elements(By.CSS_SELECTOR, "p.product-description-content")
        texts = [p.text.strip() for p in ps if p.text and p.text.strip()]

        if texts:
            return "\n".join(texts)

        container = driver.find_element(By.CSS_SELECTOR, ".product-description-section")
        txt = container.text.strip()
        return txt if txt else None

    except Exception:
        return None

In [7]:
def get_review_date(review_el):
    try:
        date_el = review_el.find_element(By.CSS_SELECTOR, ".review-info-detail .detail-item.date")
        spans = date_el.find_elements(By.CSS_SELECTOR, "span")
        if len(spans) >= 3:
            return parse_tr_date(spans[0].text, spans[1].text, spans[2].text)

        txt = safe_text(date_el)
        m = re.search(r"(\d{1,2})\s+([A-Za-zÇĞİÖŞÜçğıöşü]+)\s+(\d{4})", txt)
        if m:
            return parse_tr_date(m.group(1), m.group(2), m.group(3))
    except Exception:
        pass
    return None

In [8]:
def get_rating_from_padding(review_el):
    try:
        star_container = review_el.find_element(By.CSS_SELECTOR, '[data-testid="star-rating"]')
        full = star_container.find_element(By.CSS_SELECTOR, ".star-rating-full-star")

        container_w = float(star_container.size["width"])
        style = full.get_attribute("style") or ""
        m = re.search(r"padding-inline-end:\s*([\d.]+)px", style)
        if not m or container_w <= 0:
            return None

        pad = float(m.group(1))
        filled = max(0.0, container_w - pad)
        rating = 5.0 * (filled / container_w)

        rating = round(rating * 2) / 2  # 0.5 step
        return max(0.0, min(5.0, rating))
    except Exception:
        return None

In [9]:
def expand_read_more_inside_reviews(driver):
    try:
        buttons = driver.find_elements(By.CSS_SELECTOR, "a.read-more")
        for b in buttons:
            try:
                if b.is_displayed():
                    driver.execute_script("arguments[0].click();", b)
                    time.sleep(0.05)
            except Exception:
                pass
    except Exception:
        pass

In [10]:
def extract_one_review(review_el):
    body = try_find_text(review_el, [
        "span.review-comment",
        ".review-comment span",
        ".review-comment",
    ])

    return {
        "title": None,
        "rating": get_rating_from_padding(review_el),
        "body": body,
        "review_date": get_review_date(review_el)
    }


In [11]:
def load_all_reviews(driver, target_reviews=80, wait_sec=15, max_scrolls=60, sleep_between=0.7):
    wait = WebDriverWait(driver, wait_sec)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.review")))

    last_count = 0
    stable_rounds = 0

    for _ in range(max_scrolls):
        cards = driver.find_elements(By.CSS_SELECTOR, "div.review")
        count = len(cards)

        if count >= target_reviews:
            break

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(sleep_between)

        cards2 = driver.find_elements(By.CSS_SELECTOR, "div.review")
        count2 = len(cards2)

        if count2 >= target_reviews:
            break

        if count2 == last_count:
            stable_rounds += 1
        else:
            stable_rounds = 0
            last_count = count2

        if stable_rounds >= 3:
            break

In [12]:
def scrape_reviews_to_df(driver, product_url, review_page_url, review_summary, 
                         product_title, product_description, target_reviews=80):
    load_all_reviews(driver, target_reviews=target_reviews)

    # Expand Reviews
    expand_read_more_inside_reviews(driver)

    cards = driver.find_elements(By.CSS_SELECTOR, "div.review")[:target_reviews]

    rows = []
    for i, card in enumerate(cards):
        try:
            row = extract_one_review(card)
        except se.StaleElementReferenceException:
            cards2 = driver.find_elements(By.CSS_SELECTOR, "div.review")[:target_reviews]
            if i < len(cards2):
                row = extract_one_review(cards2[i])
            else:
                continue

        row["product_url"] = product_url
        row["source"] = "trendyol"
        row["review_summary"] = review_summary

        row["product_title"] = product_title
        row["product_description"] = product_description
        
        rows.append(row)

    df = pd.DataFrame(rows).drop_duplicates()
    turkey_tz = pytz.timezone("Europe/Istanbul")
    df["collected_at"] = datetime.now(turkey_tz).strftime("%Y-%m-%d %H:%M")
    return df

In [13]:
def get_ai_review_summary(driver, wait_sec=6):
    try:
        wait = WebDriverWait(driver, wait_sec)
        summary_section = wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "section.ai-review-text[data-testid='ai-review-summary-text']")
            )
        )

        # Try 'read more'
        try:
            more_btn = summary_section.find_element(By.CSS_SELECTOR, "button.ai-summary-custom-ellipsis")
            if more_btn.is_displayed():
                driver.execute_script("arguments[0].click();", more_btn)
                time.sleep(0.2)
        except Exception:
            pass

        return summary_section.text.strip()
    except Exception:
        return None

In [14]:
def go_to_reviews_page_and_sort_newest(driver, product_url, wait):
    # ürün sayfası
    driver.get(product_url)

    # onboarding "Anladım"
    click_if_present(driver, ".onboarding__default-renderer-primary-button", wait_sec=4)

    # ürün bilgileri
    product_title = get_product_title(driver)
    product_description = get_product_description(driver)

    # review linki
    review_tag = wait.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "a.reviews-summary-reviews-detail"))
    )
    review_page_url = review_tag.get_attribute("href")
    driver.get(review_page_url)

    # dropdown aç
    btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button.sort-dropdown-button")))
    driver.execute_script("arguments[0].click();", btn)

    # newest seç
    opt = wait.until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, 'div.sort-dropdown-item[data-testid="sort-option-newest"]'))
    )
    driver.execute_script("arguments[0].click();", opt)

    return review_page_url, product_title, product_description


In [15]:
def scrape_many_products(products, target_reviews=80, headless=False):
    driver = make_driver(detach=True, headless=headless)
    wait = WebDriverWait(driver, 15)

    all_dfs = []

    for idx, product_url in enumerate(products, 1):
        try:
            print(f"[{idx}/{len(products)}] Scraping: {product_url}")

            reviews_url, product_title, product_description = go_to_reviews_page_and_sort_newest(driver, product_url, wait)
            summary = get_ai_review_summary(driver)
            
            df = scrape_reviews_to_df(
                driver,
                product_url=product_url,
                review_page_url=reviews_url,
                review_summary=summary,
                product_title=product_title,
                product_description=product_description,
                target_reviews=target_reviews
            )
            all_dfs.append(df)

            print(f"  -> got {len(df)} reviews")

        except Exception as e:
            print(f"  !! FAILED: {product_url} | {type(e).__name__}: {e}")
            continue

    final_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame(
        columns=[
            "title","rating","body","review_date",
            "product_title","product_description",
            "product_url","reviews_url","source","review_summary","collected_at"
        ]
    )

    return driver, final_df

In [16]:
products = [
    "https://www.trendyol.com/fantom/pratic-s-p1200-toz-torbasiz-dikey-supurge-antrasit-p-35509789?boutiqueId=61&merchantId=110463",
    "https://www.trendyol.com/dokuru/cilek-kuru-meyve-cipsi-dondurularak-kurutulmus-freeze-dried-citir-cilek-p-739045260"
]

driver, df_all = scrape_many_products(products, target_reviews=80, headless=False)

[1/2] Scraping: https://www.trendyol.com/fantom/pratic-s-p1200-toz-torbasiz-dikey-supurge-antrasit-p-35509789?boutiqueId=61&merchantId=110463
  -> got 80 reviews
[2/2] Scraping: https://www.trendyol.com/dokuru/cilek-kuru-meyve-cipsi-dondurularak-kurutulmus-freeze-dried-citir-cilek-p-739045260
  -> got 80 reviews


In [15]:
pd.set_option("display.max_colwidth", None)

In [18]:
df_all.tail()

,title,rating,body,review_date,product_url,source,review_summary,product_title,product_description,collected_at
155,None,1.0,"Şeftali,kavun,mango,çilek aldım hiç güzel deği...",2026-02-14,https://www.trendyol.com/dokuru/cilek-kuru-mey...,trendyol,None,dokuru Çilek Kuru Meyve Cipsi - Dondurularak K...,"Freeze-dry işlemi, meyve ve sebzeleri uzun sür...",2026-02-18 22:05
156,None,1.0,tozdan yapılmış plastik gibi tadı vardı ben hi...,2026-02-14,https://www.trendyol.com/dokuru/cilek-kuru-mey...,trendyol,None,dokuru Çilek Kuru Meyve Cipsi - Dondurularak K...,"Freeze-dry işlemi, meyve ve sebzeleri uzun sür...",2026-02-18 22:05
157,None,5.0,hepsinin tadi mukemmel çilek benlik diil wma d...,2026-02-14,https://www.trendyol.com/dokuru/cilek-kuru-mey...,trendyol,None,dokuru Çilek Kuru Meyve Cipsi - Dondurularak K...,"Freeze-dry işlemi, meyve ve sebzeleri uzun sür...",2026-02-18 22:05
158,None,1.0,Anlatıldgı kadar abartmaları kadar hiç begenme...,2026-02-14,https://www.trendyol.com/dokuru/cilek-kuru-mey...,trendyol,None,dokuru Çilek Kuru Meyve Cipsi - Dondurularak K...,"Freeze-dry işlemi, meyve ve sebzeleri uzun sür...",2026-02-18 22:05
159,None,2.0,Fenomenlerin önerdiği hiç bir şeyi bir daha as...,2026-02-14,https://www.trendyol.com/dokuru/cilek-kuru-mey...,trendyol,None,dokuru Çilek Kuru Meyve Cipsi - Dondurularak K...,"Freeze-dry işlemi, meyve ve sebzeleri uzun sür...",2026-02-18 22:05


In [20]:
df_all.loc[0]['product_description']

'Fantom Pratik 1200 Süpürge 500 W Ürün Detay Ve Özellikleri Ekstra hafif, kullanırken yormaz Ergonomik, cana yakın, kompakt tasarım Küçük gövde dev Duvara monte edilebilme özelliği ile kapı arkası gibi yerlere kolaylıkla yerleştirilebilir Küçük ve hafif gövdesi sayesinde en dar alanlara bile kolayca sığabilir Günlük temizlikte idealdir Küçük iplikler, hafif tüyler ve kırıntılar için zahmetsiz kullanım sağlar Evcil hayvan bulunan evler için idealdir\nkedi, köpek tüylerini toplar Toz torbasız Taşıma kolaylığı ile her an elinizin altındadır Avrupa CE kalite onayı P1200 Pratic olarak da bilinir Ürun açılmamıs ambalajında gönderilir.'